# Create multi-simulation dataset (Linea 2)
Converts the 4 scaled-hydrograph SFINCS runs (BC x 0.5, 0.75, 1.25, 1.5) onto the existing
`template_100m.pkl` and merges them into ONE training pkl with 4 events:

    database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_multisim.pkl

The 1.0x event stays OUT of this dataset — it remains the existing
`ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart` pkl, used as validation/test
(config_best_sweep_multisim.yaml with validate_on_test: True).

Per-factor pkls are also kept (train/ and test/) for per-factor evaluation later.
Same conversion code as create_dataset_100m.ipynb, Step 3.

In [ ]:
import os, sys

# Resolve the repo root robustly (works in VS Code and nbconvert, any start cwd)
try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    _here = os.getcwd()
    REPO_ROOT = _here if os.path.isdir(os.path.join(_here, 'database')) \
                else os.path.abspath(os.path.join(_here, '..'))
assert os.path.isdir(os.path.join(REPO_ROOT, 'database')), f'Not the repo root: {REPO_ROOT}'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Repo root:', REPO_ROOT)

## Config

In [ ]:
TEMPLATE_PKL = 'database/datasets/train/template_100m.pkl'

# Non-warmstart sfincs_map.nc — same grid, stores x/y as 2D arrays (source point coordinates)
SFINCS_MAP_GRID = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon/sfincs_map.nc'
)

SIM_ROOT = 'database/raw_datasets_ahr/Simulations'

# the 4 scaled runs: tag -> simulation folder
SIMS = {
    'q050': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q050',
    'q075': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q075',
    'q125': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q125',
    'q150': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q150',
}

PER_SIM_NAME = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_{tag}'
MERGED_NAME  = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_multisim'
# 1.0x dataset, only used here as reference for the Q-peak sanity check
REF_1X_PKL   = 'database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart.pkl'

OUT_ROOT = 'database/datasets'

WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR = 'u'
VY_VAR = 'v'

FORCE_REBUILD_PKL = False   # True to reconvert even if the per-factor pkl exists

for tag, folder in SIMS.items():
    sim_dir = os.path.join(SIM_ROOT, folder)
    ok = all(os.path.exists(os.path.join(sim_dir, f)) for f in ['sfincs_map.nc', 'sfincs.src', 'sfincs.dis'])
    print(f"{tag}: {'OK' if ok else 'MISSING FILES'}  {sim_dir}")

## Step 1 — Create template (if missing)
Same parameters as `build_template.py` / create_dataset_100m.ipynb: SFINCS 100 m grid as finest
level + gmsh meshes at 500/1000/2000 m (4 scales total). Skipped if the template already exists —
the multisim events MUST live on the same template as the 1.0x dataset, so only rebuild it if you
are rebuilding ALL datasets.

## Step 2 — Convert each scaled simulation onto the template
(same code as create_dataset_100m.ipynb Step 3, looped over the 4 runs)

## Step 1 — Convert each scaled simulation onto the template
(same code as create_dataset_100m.ipynb Step 3, looped over the 4 runs)

## Step 3 — Merge the 4 events into one training pkl
Sanity check: the Q peaks must scale ~0.5 / 0.75 / 1.25 / 1.5 relative to the 1.0x run,
and node_BC must be identical for every event (same mesh, same source cells).

## Step 2 — Merge the 4 events into one training pkl
Sanity check: the Q peaks must scale ~0.5 / 0.75 / 1.25 / 1.5 relative to the 1.0x run,
and node_BC must be identical for every event (same mesh, same source cells).

In [ ]:
# reference: Q peak of the 1.0x event
with open(REF_1X_PKL, 'rb') as f:
    ref_1x = pickle.load(f)
q_ref = float(ref_1x[0].BC[:, :, 1].max())
node_bc_ref = ref_1x[0].node_BC.tolist()
print(f'1.0x reference: Q peak = {q_ref:.1f} m3/s   node_BC = {node_bc_ref}')
print()

merged = []
for tag in SIMS:
    pkl_path = os.path.join(OUT_ROOT, 'train', PER_SIM_NAME.format(tag=tag) + '.pkl')
    with open(pkl_path, 'rb') as f:
        data_list = pickle.load(f)
    for data in data_list:
        q_peak = float(data.BC[:, :, 1].max())
        wd_peak = float(data.WD.max())
        same_bc = data.node_BC.tolist() == node_bc_ref
        print(f'{tag}: Q peak = {q_peak:8.1f} m3/s (ratio {q_peak/q_ref:.3f})   '
              f'WD peak = {wd_peak:.3f} m   node_BC identical: {same_bc}')
        assert same_bc, f'{tag}: node_BC differs from the 1.0x event!'
    merged += data_list

out_path = os.path.join(OUT_ROOT, 'train', MERGED_NAME + '.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(merged, f)
print(f'\nSaved {len(merged)} events -> {out_path}')
print('Train with config_best_sweep_multisim.yaml (validation/test = the held-out 1.0x event).')